# Gastrula-to-Pup multimodal training with RegularizedMultimodalVI

This notebook trains a **3-modality model** (total RNA + spliced + unspliced) on the
Qiu et al. 2024 mouse prenatal development dataset using `RegularizedMultimodalVI`.

**Prerequisites**: Run NB1d (`01d_cell_filtering.ipynb`) first to produce
`gastrula_to_pup_filtered_qc.h5mu`.

**Key features**:
- Purpose-driven covariate keys designed for sci-RNA-seq3 batch structure
- Papermill parameters for experiment sweeps
- W&B experiment tracking (optional)
- Decoder attribution analysis

In [ ]:
import os
import pathlib
import time

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import mudata as mu
import torch
from matplotlib import rcParams

import regularizedvi
import scvi

scvi.settings.progress_bar_style = "tqdm"
torch.set_float32_matmul_precision("high")
rcParams["pdf.fonttype"] = 42

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
results_folder = "results/gastrula_to_pup/"
os.makedirs(results_folder, exist_ok=True)

## 1. Load pre-filtered MuData

Load the cell-filtered MuData produced by NB1d. Each modality (rna, spliced, unspliced)
has already been subset to informative genes and QC-filtered cells.

In [ ]:
h5mu_path = os.path.join(DATA_DIR, "gastrula_to_pup_filtered_qc.h5mu")
mdata = mu.read_h5mu(h5mu_path)
print(mdata)
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}")

## 2. Model setup — covariate design for sci-RNA-seq3

sci-RNA-seq3 uses **3-level combinatorial indexing**, each introducing distinct
technical effects. We assign each effect to the appropriate covariate key:

| Key | Value | Rationale |
|-----|-------|----------|
| `ambient_covariate_keys` | `["embryo_id", "pcr_well"]` | RT-step + protease-step ambient |
| `nn_conditioning_covariate_keys` | `["experimental_batch"]` | Multiplicative reagent/sequencing biases |
| `feature_scaling_covariate_keys` | `None` (inherits) | Per-feature scaling mirrors nn_conditioning |
| `dispersion_key` | `"embryo_experiment"` | Composite of embryo_id x experimental_batch |
| `library_size_key` | `"pcr_well"` | Sequencing budget per PCR well |
| `encoder_covariate_keys` | `False` | No covariates in encoder (prevent batch leakage) |

**Why not `embryo_id` in `nn_conditioning`?** Embryos span developmental stages —
using embryo_id as multiplicative covariate would absorb biological variation.

In [ ]:
regularizedvi.RegularizedMultimodalVI.setup_mudata(
    mdata,
    ambient_covariate_keys=["embryo_id", "pcr_well"],
    nn_conditioning_covariate_keys=["experimental_batch"],
    feature_scaling_covariate_keys=None,
    dispersion_key="embryo_experiment",
    library_size_key="pcr_well",
    encoder_covariate_keys=False,
)

## 3. Architecture + hyperparameters

Large architecture for a ~7M-cell, 3-modality dataset:
- RNA: 2048 hidden, 512 latent (dominant modality)
- Spliced/Unspliced: 512 hidden, 128 latent each
- Total latent dim = 512 + 128 + 128 = 768

All 3 modalities get additive background correction and feature scaling (symmetric design).

In [ ]:
additive_bg_prior_alpha = 1.0
additive_bg_prior_beta = 100.0
regularise_background = 1
compute_pearson = 1
use_feature_scaling = 1
skip_training = 0
max_epochs = 25
early_stopping_patience = 5
batch_size = 1024
wandb_project = None
wandb_name = None
wandb_entity = None
wandb_notes = None
wandb_group = None

In [ ]:
from regularizedvi.utils import coerce_papermill_params, finish_wandb, log_figure_to_wandb, setup_wandb_logger

params = coerce_papermill_params(
    additive_bg_prior_alpha=(additive_bg_prior_alpha, float),
    additive_bg_prior_beta=(additive_bg_prior_beta, float),
    regularise_background=(regularise_background, bool),
    compute_pearson=(compute_pearson, bool),
    use_feature_scaling=(use_feature_scaling, bool),
    skip_training=(skip_training, bool),
    max_epochs=(max_epochs, int),
    early_stopping_patience=(early_stopping_patience, int),
    batch_size=(batch_size, int),
    wandb_project=(wandb_project, "str_or_none"),
    wandb_name=(wandb_name, "str_or_none"),
    wandb_entity=(wandb_entity, "str_or_none"),
    wandb_notes=(wandb_notes, "str_or_none"),
    wandb_group=(wandb_group, "str_or_none"),
)
additive_bg_prior_alpha = params["additive_bg_prior_alpha"]
additive_bg_prior_beta = params["additive_bg_prior_beta"]
regularise_background = params["regularise_background"]
compute_pearson = params["compute_pearson"]
use_feature_scaling = params["use_feature_scaling"]
skip_training = params["skip_training"]
max_epochs = params["max_epochs"]
early_stopping_patience = params["early_stopping_patience"]
batch_size = params["batch_size"]
wandb_project = params["wandb_project"]
wandb_name = params["wandb_name"]
wandb_entity = params["wandb_entity"]
wandb_notes = params["wandb_notes"]
wandb_group = params["wandb_group"]

wandb_loggers, wandb_run = setup_wandb_logger(
    wandb_project=wandb_project,
    wandb_name=wandb_name,
    wandb_entity=wandb_entity,
    wandb_notes=wandb_notes,
    wandb_group=wandb_group,
    config={
        "additive_bg_prior_alpha": additive_bg_prior_alpha,
        "additive_bg_prior_beta": additive_bg_prior_beta,
        "regularise_background": regularise_background,
        "compute_pearson": compute_pearson,
        "use_feature_scaling": use_feature_scaling,
        "max_epochs": max_epochs,
        "n_hidden": {"rna": 2048, "spliced": 512, "unspliced": 512},
        "n_latent": {"rna": 512, "spliced": 128, "unspliced": 128},
        "n_layers": 1,
        "latent_mode": "concatenation",
    },
    results_folder=results_folder,
)

In [ ]:
model = regularizedvi.RegularizedMultimodalVI(
    mdata,
    n_hidden={"rna": 2048, "spliced": 512, "unspliced": 512},
    n_latent={"rna": 512, "spliced": 128, "unspliced": 128},
    n_layers=1,
    latent_mode="concatenation",
    additive_background_modalities=["rna", "spliced", "unspliced"],
    feature_scaling_modalities=["rna", "spliced", "unspliced"] if use_feature_scaling else [],
    dispersion="gene-batch",
    regularise_dispersion=True,
    additive_bg_prior_alpha=additive_bg_prior_alpha,
    additive_bg_prior_beta=additive_bg_prior_beta,
    regularise_background=regularise_background,
    compute_pearson=compute_pearson,
)

print(model)
print(f"\nTotal latent dim: {model.module.total_latent_dim}")
for name in model.module.modality_names:
    print(f"  Z_{name} dim: {model.module.n_latent_dict[name]}")
print(f"\nRegularise background: {regularise_background}")
if regularise_background:
    print(f"  Prior: Gamma({additive_bg_prior_alpha}, {additive_bg_prior_beta})")
    print(f"  Prior mean: {additive_bg_prior_alpha / additive_bg_prior_beta:.4f}")
print(f"Compute Pearson: {compute_pearson}")

## 4. Training

With ~7M cells and `batch_size=1024`, one epoch ~ 7K steps.
Early stopping monitors `elbo_validation` with checkpoints saved every epoch.

In [ ]:
if skip_training:
    ref_run_name = f"{results_folder}/model"
    model = regularizedvi.RegularizedMultimodalVI.load(ref_run_name, adata=mdata)
    print(f"Loaded model from {ref_run_name}")
    n_epochs = len(model.history_["elbo_train"])
    print(f"Model was trained for {n_epochs} epochs")
else:
    from scvi.train import SaveCheckpoint

    checkpoint_cb = SaveCheckpoint(
        dirpath=f"{results_folder}/checkpoints",
        every_n_epochs=1,
        save_top_k=-1,
        filename="epoch-{epoch}",
    )

    t0 = time.time()
    model.train(
        check_val_every_n_epoch=1,
        train_size=0.9,
        max_epochs=max_epochs,
        batch_size=batch_size,
        early_stopping=True,
        early_stopping_patience=early_stopping_patience,
        early_stopping_monitor="elbo_validation",
        enable_checkpointing=True,
        callbacks=[checkpoint_cb],
        logger=wandb_loggers,
    )
    elapsed = time.time() - t0

    n_epochs = len(model.history_["elbo_train"])
    print(f"\nTraining finished: {n_epochs} epochs in {elapsed / 60:.1f} min ({elapsed / n_epochs:.2f} s/epoch)")
    print(f"Throughput: {mdata.n_obs * n_epochs / elapsed:.0f} cells*epochs/s")

In [ ]:
ref_run_name = f"{results_folder}/model"
model.save(ref_run_name, overwrite=True)

In [ ]:
fig = model.plot_training_diagnostics(skip_epochs=min(5, n_epochs // 2))
log_figure_to_wandb("training_diagnostics", fig)
plt.show()

## 5. Latent space + UMAP

With `latent_mode='concatenation'`, Z = [Z_rna; Z_spliced; Z_unspliced].
`compute_latent_umap` extracts per-modality slices and computes separate UMAPs.

In [ ]:
ref_adata = mdata["rna"]
model.compute_latent_umap(ref_adata)
print("Computed joint + per-modality latent UMAPs")

### Per-modality UMAP comparison

Compare joint Z, RNA Z, spliced Z, and unspliced Z embeddings coloured by
developmental stage and cell type.

In [ ]:
color = ["day", "embryo_id", "experimental_batch", "major_trajectory", "celltype_update"]
color = [c for c in color if c in ref_adata.obs.columns]

fig = model.plot_umap_comparison(
    ref_adata,
    color=color,
    umap_keys=[
        ("X_umap_joint", "Joint Z"),
        ("X_umap_rna", "RNA Z"),
        ("X_umap_spliced", "Spliced Z"),
        ("X_umap_unspliced", "Unspliced Z"),
    ],
)
log_figure_to_wandb("umap_4xN", fig)
plt.show()

### Load a previously trained model (optional)

Uncomment below to load from a saved checkpoint instead of retraining.

In [ ]:
# model = regularizedvi.RegularizedMultimodalVI.load(f"{results_folder}/checkpoints/epoch-XXXX/", adata=mdata)

## 6. Decoder attribution analysis (Jacobian-based)

Compute the **mean absolute Jacobian** `|d(px_rate)/d(z)|` per latent dimension for
each modality's decoder. This reveals which parts of the shared
Z = [Z_rna; Z_spliced; Z_unspliced] inform each modality.

**What to look for**: If each decoder uses primarily its own Z dims, the modalities
learned independent representations. If spliced/unspliced decoders rely on Z_rna dims,
they are **free-riding** on RNA-derived information.

In [ ]:
attribution, fig_bar = model.plot_modality_attribution(batch_size=256)
log_figure_to_wandb("attribution_bar", fig_bar)
plt.show()

### Attribution-weighted UMAPs

The weighted Z (`z * attribution`) emphasises latent dimensions each decoder uses.
Differences between modality attribution UMAPs reveal cells driven by different modalities.

In [ ]:
attribution = model.store_attribution_results(ref_adata, attribution=attribution)
model.compute_attribution_umap(ref_adata)
print("Computed attribution-weighted UMAPs")

In [ ]:
color_attr = ["experimental_batch", "celltype_update", "major_trajectory"]
color_attr = [c for c in color_attr if c in ref_adata.obs.columns]

fig = model.plot_umap_comparison(
    ref_adata,
    color=color_attr,
    umap_keys=[
        ("X_umap_joint", "Joint Z"),
        ("X_umap_rna", "RNA Z"),
        ("X_umap_spliced", "Spliced Z"),
        ("X_umap_unspliced", "Unspliced Z"),
        ("X_umap_attr_rna", "RNA attribution Z"),
        ("X_umap_attr_spliced", "Spliced attribution Z"),
        ("X_umap_attr_unspliced", "Unspliced attribution Z"),
    ],
    figsize_per_panel=(5, 5),
)
log_figure_to_wandb("umap_summary_7xN", fig)
plt.show()

## 7. Marker gene visualisation

Overlay developmental marker genes on the Joint Z UMAP to verify the latent space
captures expected biology.

In [ ]:
ref_adata.obsm["X_umap"] = ref_adata.obsm["X_umap_joint"]

_nb_dir = pathlib.Path("docs/notebooks")
if not _nb_dir.exists():
    _nb_dir = pathlib.Path(".")
marker_df = pd.read_csv(_nb_dir / "known_marker_genes.csv")
categories = marker_df.groupby("category", sort=False)["gene"].apply(lambda x: list(dict.fromkeys(x)))

gene_col = "gene_short_name" if "gene_short_name" in ref_adata.var.columns else "SYMBOL"
rcParams["figure.figsize"] = 6, 6

for category, gene_list in categories.items():
    present = [g for g in gene_list if g in ref_adata.var[gene_col].values]
    if not present:
        continue

    gene_idx = ref_adata.var[gene_col].isin(present)
    counts_col = "total_counts" if "total_counts" in ref_adata.obs.columns else "GEX_n_counts"
    selected_expr = ref_adata[:, gene_idx].X.multiply(1.0 / ref_adata.obs[[counts_col]].values)
    selected_expr = selected_expr.toarray() * 1e4

    col_names = [f"{m} normalised" for m in present]
    ref_adata.obs[col_names] = selected_expr

    print(f"\n{'=' * 60}")
    print(f"{category} ({len(present)} genes)")
    print(f"{'=' * 60}")

    sc.pl.umap(
        ref_adata,
        color=col_names,
        color_map="RdPu",
        ncols=5,
        size=3,
        vmin=0,
        vmax="p99.99",
        use_raw=False,
        legend_fontsize=8,
        title=[f"{m}" for m in present],
    )

## 8. Save outputs

In [ ]:
output_dir = f"{ref_run_name}/outputs/"
saved = model.save_analysis_outputs(
    output_dir,
    ref_adata,
    save_attribution=True,
    attribution=attribution,
)
print(f"Saved {len(saved)} files to {output_dir}")

In [ ]:
finish_wandb()

## Summary

This notebook demonstrated **RegularizedMultimodalVI** with 3 modalities on the
Qiu et al. 2024 gastrula-to-pup dataset:

1. **Purpose-driven covariate keys** — properly assigned to sci-RNA-seq3 batch hierarchy
2. **3-modality architecture** — total RNA (512 latent), spliced (128), unspliced (128)
   with symmetric additive background and feature scaling
3. **Per-modality diagnostics** — reconstruction loss, Pearson r, KL divergence per modality
4. **4 UMAP views** — joint Z, RNA Z, spliced Z, unspliced Z
5. **Decoder attribution** — reveals which modalities each decoder relies on
6. **Marker genes** — developmental marker gene overlay validates biological content

### Alternative Z strategies

```python
# Weighted mean (MultiVI-style)
model = RegularizedMultimodalVI(mdata, latent_mode='weighted_mean', n_latent=64)

# Single encoder
model = RegularizedMultimodalVI(mdata, latent_mode='single_encoder')
```